<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/04-pythonic-tools/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: A Hyperparameter Search Framework

Every model has settings you choose before training: the learning rate, the number of neighbours, the amount of regularization. These are **hyperparameters**, and finding good values for them is a big part of practical ML. Tools like Optuna, Ray Tune and scikit-learn's `GridSearchCV` automate it.

In this project you'll build a small version of one of those tools, using everything from this module:

- a **registry decorator** so models can be added by name, without touching the search code
- **`itertools.product`** and **generators** to expand a search space into individual trials, lazily
- **`functools.cache`** so the dataset is only built once, however many trials use it
- a **context manager** that times each trial and records failures instead of crashing the whole search
- a **generator** that streams results as each trial finishes, so you can watch progress live
- **`itertools.groupby`** to find the best configuration of each model
- **type hints** throughout

At the end, you'll have a reusable search tool and a leaderboard comparing four kinds of model, and you'll evaluate the winner on a test set it has never seen.

## Step 1: The data

We'll generate two overlapping clouds of 2D points, one per class. Because they overlap, no model can get 100%, which makes it a realistic test: the question is how close each model gets to the best possible score.

Like most real measurements, the feature values sit well away from zero: around 2 to 4 for the first feature and 10 to 11 for the second. That detail will turn out to matter.

The loader is wrapped in `functools.cache`. A search may run dozens of trials, and they should all use exactly the same data without regenerating it each time. The `print` inside shows when the function body actually runs.

The function returns tuples rather than lists. A cached function hands the *same* object to every caller, so if one trial modified a list it got back, every later trial would see the modified data. Tuples can't be modified, which rules that out.

In [1]:
import functools
import itertools
import math
import random
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Iterator, Protocol

Features = tuple[float, ...]
Dataset = tuple[tuple[Features, ...], tuple[int, ...]]


@functools.cache
def load_split(name: str) -> Dataset:
    sizes = {"train": 300, "val": 150, "test": 300}
    seeds = {"train": 0, "val": 1, "test": 2}
    print(f"  (generating {name} split)")
    rng = random.Random(seeds[name])
    X, y = [], []
    for _ in range(sizes[name]):
        label = rng.random() < 0.5
        center = (4.0, 11.0) if label else (2.0, 10.0)
        X.append((rng.gauss(center[0], 1.0), rng.gauss(center[1], 1.0)))
        y.append(int(label))
    return tuple(X), tuple(y)


X_train, y_train = load_split("train")
X_train, y_train = load_split("train")     # cached: no message the second time
print(len(X_train), "training examples, first one:", X_train[0], y_train[0])
print(load_split.cache_info())

  (generating train split)
300 training examples, first one: (2.0521919882826083, 8.956591025799426) 0
CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)


## Step 2: A model registry

The search code shouldn't need to know which models exist. It should be able to say "build the model called `knn` with `k=5`" and get one. That's the `Registry` from exercise 10:

In [2]:
class Registry:
    def __init__(self):
        self._items: dict[str, type] = {}

    def register(self, name: str):
        def decorator(obj):
            if name in self._items:
                raise ValueError(f"{name!r} is already registered")
            self._items[name] = obj
            return obj
        return decorator

    def build(self, name: str, **kwargs):
        if name not in self._items:
            raise KeyError(f"{name!r} is not registered. Available: {self.names()}")
        return self._items[name](**kwargs)

    def names(self) -> list[str]:
        return sorted(self._items)


MODELS = Registry()

Every model follows the same interface: `fit(X, y)` returns `self`, and `predict(X)` returns a list of labels. We write that down as a `Protocol`, so it's documented and checkable, without forcing the models to inherit from anything.

In [3]:
class Model(Protocol):
    def fit(self, X: tuple[Features, ...], y: tuple[int, ...]) -> "Model": ...
    def predict(self, X: tuple[Features, ...]) -> list[int]: ...

## Step 3: The models

Three models, each registered with one decorator line.

**Majority** always predicts the most common class. It's the baseline every other model has to beat.

In [4]:
@MODELS.register("majority")
class Majority:
    def fit(self, X, y):
        self.label_ = max(set(y), key=y.count)
        return self

    def predict(self, X):
        return [self.label_] * len(X)

**k-nearest neighbours**, as in the Module 1 mini project. It refuses a `k` bigger than the training set, with a clear error. We'll see that error get caught by the search later.

In [5]:
@MODELS.register("knn")
class KNN:
    def __init__(self, k: int = 5):
        self.k = k

    def fit(self, X, y):
        if self.k > len(X):
            raise ValueError(f"k={self.k} is larger than the training set ({len(X)} examples)")
        self.X_, self.y_ = X, y
        return self

    def predict(self, X):
        preds = []
        for x in X:
            dists = sorted(
                (sum((a - b) ** 2 for a, b in zip(x, xt)), label)
                for xt, label in zip(self.X_, self.y_)
            )
            votes = [label for _, label in dists[:self.k]]
            preds.append(max(set(votes), key=votes.count))
        return preds

**Logistic regression**, trained with gradient descent. It computes a weighted sum of the features, squashes it into a probability between 0 and 1 with the sigmoid function, and predicts class 1 if the probability is at least 0.5. Training nudges the weights to make the probabilities match the labels. `l2` adds a penalty for large weights, a common form of regularization.

The sigmoid is written in two branches to avoid overflow: `math.exp(-z)` for a large negative `z` would be `exp` of a huge number, which raises an `OverflowError`. Rearranging the formula for negative `z` keeps every `exp` call small.

In [6]:
def sigmoid(z: float) -> float:
    if z >= 0:
        return 1.0 / (1.0 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1.0 + ez)


@MODELS.register("logreg")
class LogisticRegression:
    def __init__(self, lr: float = 0.1, epochs: int = 100, l2: float = 0.0):
        self.lr, self.epochs, self.l2 = lr, epochs, l2

    def _score(self, x: Features) -> float:
        return sum(w * xi for w, xi in zip(self.w_, x)) + self.b_

    def fit(self, X, y):
        n, d = len(X), len(X[0])
        self.w_, self.b_ = [0.0] * d, 0.0
        for _ in range(self.epochs):
            grad_w, grad_b = [0.0] * d, 0.0
            for x, target in zip(X, y):
                error = sigmoid(self._score(x)) - target
                for j in range(d):
                    grad_w[j] += error * x[j]
                grad_b += error
            for j in range(d):
                self.w_[j] -= self.lr * (grad_w[j] / n + self.l2 * self.w_[j])
            self.b_ -= self.lr * grad_b / n
        return self

    def predict(self, X):
        return [1 if self._score(x) >= 0 else 0 for x in X]


print(MODELS.names())

['knn', 'logreg', 'majority']


In `predict`, `self._score(x) >= 0` is the same as `sigmoid(...) >= 0.5`, since the sigmoid crosses 0.5 exactly at 0. It skips an unnecessary `exp` call for every prediction.

Let's make sure each model works on its own before searching:

In [7]:
X_val, y_val = load_split("val")


def accuracy(y_true, y_pred) -> float:
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


for name, params in [("majority", {}), ("knn", {"k": 5}), ("logreg", {"lr": 0.1})]:
    model = MODELS.build(name, **params).fit(X_train, y_train)
    print(f"{name:8s} val accuracy: {accuracy(y_val, model.predict(X_val)):.1%}")

  (generating val split)
majority val accuracy: 49.3%
knn      val accuracy: 86.0%
logreg   val accuracy: 77.3%


## Step 4: The search space

A search space says which values to try for each model:

In [8]:
SEARCH_SPACE: dict[str, dict[str, list]] = {
    "majority": {},
    "knn": {"k": [1, 5, 15, 51, 500]},
    "logreg": {"lr": [0.01, 0.1, 1.0], "epochs": [10, 100], "l2": [0.0, 0.1]},
}

Notice `k=500`. The training set only has 300 examples, so that trial will fail. In a real search, some configurations always fail, from bad combinations of settings, numerical problems or running out of memory. The framework has to cope with that.

Now we expand the space into individual trials. `expand_grid` uses `itertools.product` exactly as in the lesson. Both functions are generators, so nothing is built up front. Trials are produced one at a time as the search asks for them, which matters when a search space has thousands of combinations.

In [9]:
def expand_grid(grid: dict[str, list]) -> Iterator[dict]:
    keys = list(grid)
    for values in itertools.product(*grid.values()):
        yield dict(zip(keys, values))


def all_trials(space: dict[str, dict[str, list]]) -> Iterator[tuple[str, dict]]:
    for model_name, grid in space.items():
        for params in expand_grid(grid):
            yield model_name, params


trials = list(all_trials(SEARCH_SPACE))
print(len(trials), "trials")
for t in trials[:4]:
    print(" ", t)

18 trials
  ('majority', {})
  ('knn', {'k': 1})
  ('knn', {'k': 5})
  ('knn', {'k': 15})


`majority` has an empty grid, and `itertools.product()` with no arguments produces exactly one empty tuple. So `expand_grid({})` yields one empty dict, meaning "run once with default settings". Edge cases like this often fall out of the maths for free.

## Step 5: Recording each trial

Each trial produces a result. A frozen dataclass from Module 3 is a good fit: results shouldn't change after they're recorded.

In [10]:
@dataclass(frozen=True)
class TrialResult:
    model: str
    params: dict
    val_accuracy: float | None
    seconds: float
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.error is None

    def describe(self) -> str:
        params = ", ".join(f"{k}={v}" for k, v in self.params.items()) or "defaults"
        if not self.ok:
            return f"{self.model:13s} {params:30s} FAILED: {self.error}"
        return f"{self.model:13s} {params:30s} acc={self.val_accuracy:.1%}  ({self.seconds * 1000:.0f} ms)"

Now the context manager. It times the block, and if the block raises, it records the error and **suppresses** it, so the search can move on to the next trial.

The lesson said context managers usually shouldn't swallow exceptions. This is one of the deliberate exceptions to that rule, and it's safe because the error isn't lost: it's recorded in the result and shown on the leaderboard. Swallowing an error silently would be a bug. Catching it, writing it down and carrying on is a design decision.

In [11]:
@contextmanager
def trial_recorder(model_name: str, params: dict, results: list[TrialResult]):
    record = {"val_accuracy": None}
    start = time.perf_counter()
    try:
        yield record
    except Exception as e:
        results.append(TrialResult(model_name, params, None,
                                   time.perf_counter() - start, f"{type(e).__name__}: {e}"))
    else:
        results.append(TrialResult(model_name, params, record["val_accuracy"],
                                   time.perf_counter() - start))

The block fills in `record["val_accuracy"]`. The `else` branch of a `try` runs only if no exception happened, so successful and failed trials are recorded in separate branches.

`except Exception` is deliberately broad here, since we want to survive *any* failure in a trial. But it doesn't catch `KeyboardInterrupt`, which isn't a subclass of `Exception`, so pressing stop still stops the whole search, as it should.

## Step 6: The search loop

`run_search` is a generator. It yields each result as soon as the trial finishes, so the caller can print progress live, stop early, or save results as they come in.

In [12]:
def run_search(space, train: Dataset, val: Dataset) -> Iterator[TrialResult]:
    X_tr, y_tr = train
    X_v, y_v = val
    for model_name, params in all_trials(space):
        results: list[TrialResult] = []
        with trial_recorder(model_name, params, results) as record:
            model = MODELS.build(model_name, **params)
            model.fit(X_tr, y_tr)
            record["val_accuracy"] = accuracy(y_v, model.predict(X_v))
        yield results[0]


all_results: list[TrialResult] = []
start = time.perf_counter()
for result in run_search(SEARCH_SPACE, load_split("train"), load_split("val")):
    all_results.append(result)
    print(result.describe())

print(f"\n{len(all_results)} trials in {time.perf_counter() - start:.1f} s")

majority      defaults                       acc=49.3%  (0 ms)
knn           k=1                            acc=85.3%  (39 ms)


knn           k=5                            acc=86.0%  (39 ms)


knn           k=15                           acc=86.7%  (43 ms)


knn           k=51                           acc=87.3%  (40 ms)
knn           k=500                          FAILED: ValueError: k=500 is larger than the training set (300 examples)
logreg        lr=0.01, epochs=10, l2=0.0     acc=49.3%  (3 ms)
logreg        lr=0.01, epochs=10, l2=0.1     acc=49.3%  (4 ms)


logreg        lr=0.01, epochs=100, l2=0.0    acc=76.0%  (36 ms)
logreg        lr=0.01, epochs=100, l2=0.1    acc=75.3%  (33 ms)
logreg        lr=0.1, epochs=10, l2=0.0      acc=50.7%  (4 ms)


logreg        lr=0.1, epochs=10, l2=0.1      acc=50.7%  (5 ms)
logreg        lr=0.1, epochs=100, l2=0.0     acc=77.3%  (28 ms)


logreg        lr=0.1, epochs=100, l2=0.1     acc=66.0%  (35 ms)
logreg        lr=1.0, epochs=10, l2=0.0      acc=78.0%  (3 ms)
logreg        lr=1.0, epochs=10, l2=0.1      acc=58.7%  (3 ms)


logreg        lr=1.0, epochs=100, l2=0.0     acc=65.3%  (32 ms)


logreg        lr=1.0, epochs=100, l2=0.1     acc=54.0%  (38 ms)

18 trials in 0.4 s


The `k=500` trial failed with a clear message, and the search carried on without it. The data was only generated once (the "generating" messages appeared in earlier steps, not here), thanks to the cache.

## Step 7: The leaderboard

Let's sort the successful trials by validation accuracy, breaking ties by speed, using the tuple-key trick from Module 2:

In [13]:
ok = [r for r in all_results if r.ok]
failed = [r for r in all_results if not r.ok]

leaderboard = sorted(ok, key=lambda r: (-r.val_accuracy, r.seconds))

print("Top 5")
for rank, r in enumerate(leaderboard[:5], start=1):
    print(f"{rank}. {r.describe()}")

print(f"\n{len(failed)} failed:")
for r in failed:
    print("  ", r.describe())

Top 5
1. knn           k=51                           acc=87.3%  (40 ms)
2. knn           k=15                           acc=86.7%  (43 ms)
3. knn           k=5                            acc=86.0%  (39 ms)
4. knn           k=1                            acc=85.3%  (39 ms)
5. logreg        lr=1.0, epochs=10, l2=0.0      acc=78.0%  (3 ms)

1 failed:
   knn           k=500                          FAILED: ValueError: k=500 is larger than the training set (300 examples)


And the best configuration for each model type, with `groupby`. Remember that `groupby` only groups *consecutive* items, so we sort by model name first:

In [14]:
by_model = sorted(ok, key=lambda r: r.model)
best_per_model = {
    name: max(group, key=lambda r: r.val_accuracy)
    for name, group in itertools.groupby(by_model, key=lambda r: r.model)
}

for name, r in sorted(best_per_model.items(), key=lambda kv: -kv[1].val_accuracy):
    print(r.describe())

knn           k=51                           acc=87.3%  (40 ms)
logreg        lr=1.0, epochs=10, l2=0.0      acc=78.0%  (3 ms)
majority      defaults                       acc=49.3%  (0 ms)


A few things to read from this:

- **Everything beats the baseline**, which scores about 50% because the classes are balanced. So the models are learning something real.
- **k-NN gets better as `k` grows.** With `k=1`, every noisy training point near the overlap gets its own little region, which is overfitting. Averaging over more neighbours smooths that out.
- **Logistic regression is all over the place.** Look back at the full list: depending on its settings, it scores anywhere from about 49% to 78%. With `lr=0.01` and 10 epochs it barely moves from its starting weights and predicts one class for everything. With `lr=1.0` and 100 epochs it overshoots and does worse than with 10 epochs. And even its best setting is well behind k-NN.

So is logistic regression just a bad model for this data? The classes are two round clouds, which a straight line separates well, so it *should* do fine. The problem is the one from the Module 1 mini project: the features aren't scaled. They sit far from zero, so the bias has to grow large while the weights stay small, and plain gradient descent is slow and unstable at that. k-NN only compares distances between points, so the offset doesn't bother it.

Let's test that theory, and see how easily the framework takes a new model.

## Step 8: Adding a new model

Here's the payoff of the registry. To add a model to the search, you write the class, register it, and give it a grid. The search, recording and leaderboard code doesn't change at all.

The new model is logistic regression with standardization built in: during `fit`, it learns the mean and standard deviation of each feature from the training data, rescales the features to mean 0 and standard deviation 1, then trains as before. During `predict`, it applies the *same* scaling, learned from training data, to the new data.

It inherits from `LogisticRegression` and uses `super()` from Module 3 to reuse all the training code, so only the scaling is new.

In [15]:
@MODELS.register("logreg_scaled")
class ScaledLogisticRegression(LogisticRegression):
    def fit(self, X, y):
        d = len(X[0])
        self.mean_ = [sum(x[j] for x in X) / len(X) for j in range(d)]
        self.std_ = [
            (sum((x[j] - self.mean_[j]) ** 2 for x in X) / len(X)) ** 0.5 or 1.0
            for j in range(d)
        ]
        return super().fit(self._scale(X), y)

    def _scale(self, X):
        return tuple(
            tuple((v - m) / s for v, m, s in zip(x, self.mean_, self.std_))
            for x in X
        )

    def predict(self, X):
        return super().predict(self._scale(X))


new_space = {"logreg_scaled": {"lr": [0.01, 0.1, 1.0], "epochs": [10, 100], "l2": [0.0, 0.1]}}
for result in run_search(new_space, load_split("train"), load_split("val")):
    all_results.append(result)
    print(result.describe())

logreg_scaled lr=0.01, epochs=10, l2=0.0     acc=86.7%  (3 ms)
logreg_scaled lr=0.01, epochs=10, l2=0.1     acc=86.7%  (3 ms)


logreg_scaled lr=0.01, epochs=100, l2=0.0    acc=86.7%  (25 ms)
logreg_scaled lr=0.01, epochs=100, l2=0.1    acc=86.7%  (30 ms)
logreg_scaled lr=0.1, epochs=10, l2=0.0      acc=86.7%  (5 ms)
logreg_scaled lr=0.1, epochs=10, l2=0.1      acc=86.7%  (5 ms)


logreg_scaled lr=0.1, epochs=100, l2=0.0     acc=88.0%  (38 ms)


logreg_scaled lr=0.1, epochs=100, l2=0.1     acc=87.3%  (29 ms)
logreg_scaled lr=1.0, epochs=10, l2=0.0      acc=88.0%  (3 ms)
logreg_scaled lr=1.0, epochs=10, l2=0.1      acc=87.3%  (3 ms)
logreg_scaled lr=1.0, epochs=100, l2=0.0     acc=87.3%  (26 ms)

logreg_scaled lr=1.0, epochs=100, l2=0.1     acc=87.3%  (27 ms)


That confirms it. With scaled features, logistic regression is now competitive with the best k-NN. Just as important, it's **stable**: every one of the twelve settings lands in a narrow range, compared with the wild spread from 49% to 78% before. Good preprocessing doesn't only raise the best score, it also makes a model much less sensitive to its hyperparameters, which makes everything after it easier.

`... or 1.0` in the standard deviation handles a feature that's constant in the training data: its standard deviation is 0, which is falsy, so `or` replaces it with 1.0 and avoids dividing by zero.

Here's the updated leaderboard, with the new results included:

In [16]:
ok = [r for r in all_results if r.ok]
leaderboard = sorted(ok, key=lambda r: (-r.val_accuracy, r.seconds))

for rank, r in enumerate(leaderboard[:5], start=1):
    print(f"{rank}. {r.describe()}")

1. logreg_scaled lr=1.0, epochs=10, l2=0.0      acc=88.0%  (3 ms)
2. logreg_scaled lr=0.1, epochs=100, l2=0.0     acc=88.0%  (38 ms)
3. logreg_scaled lr=1.0, epochs=10, l2=0.1      acc=87.3%  (3 ms)
4. logreg_scaled lr=1.0, epochs=100, l2=0.0     acc=87.3%  (26 ms)
5. logreg_scaled lr=1.0, epochs=100, l2=0.1     acc=87.3%  (27 ms)


## Step 9: Evaluate the winner on the test set

We chose the best configuration by looking at validation accuracy. That means the validation score is now slightly optimistic: out of 30 trials, we picked the one that happened to do best on these particular 150 points, partly by luck.

The test set gives an honest estimate, but only if it's used **once**, at the very end, for the final chosen model. That's why this step comes last, after we'd finished adding models. If we had checked the test score after step 7, then added a model and checked again, we'd have been using the test set to make decisions, and it would no longer be an honest measure. It would have become just another validation set.

In [17]:
best = leaderboard[0]
print("chosen:", best.describe())

final_model = MODELS.build(best.model, **best.params).fit(*load_split("train"))
X_test, y_test = load_split("test")
test_acc = accuracy(y_test, final_model.predict(X_test))

print(f"validation accuracy: {best.val_accuracy:.1%}")
print(f"test accuracy:       {test_acc:.1%}")

chosen: logreg_scaled lr=1.0, epochs=10, l2=0.0      acc=88.0%  (3 ms)
  (generating test split)
validation accuracy: 88.0%
test accuracy:       84.0%


`fit(*load_split("train"))` unpacks the `(X, y)` tuple into two arguments, the star unpacking from Module 2.

Look at how the test score compares to the validation score. Some gap is normal, for the reason above: picking the best of many trials on one validation set flatters the winner a little. A *large* gap would suggest the search overfit to the validation set, which usually means the validation set is too small, or too many configurations were tried. Cross-validation (in "Your turn" below) is the standard way to make the selection more reliable.

## Your turn

1. **Random search.** Write a generator `random_trials(space, n, seed)` that picks a random value for each parameter instead of trying every combination. Better still, make it infinite and use `itertools.islice` to take `n` trials. For large search spaces, random search often finds good settings much faster than a full grid, because most hyperparameters matter much less than one or two key ones.

2. **Continuous ranges.** Let the search space contain ranges like `("log_uniform", 1e-3, 1.0)` for the learning rate, and sample from them in `random_trials`. Learning rates are usually searched on a log scale, because the difference between 0.001 and 0.01 matters as much as the difference between 0.1 and 1.

3. **A time budget.** Stop the search once a given number of seconds has passed. Since `run_search` is a generator, you can do this entirely from the outside, in the loop that consumes it, without changing `run_search` at all.

4. **Cross-validation.** One validation split is noisy, especially with only 150 points. Split the training data into 5 parts, train on 4 and validate on the fifth, rotating through all five, and average the scores. Your `batched` from the exercises can do the splitting.

5. **Save the results.** Write each `TrialResult` to a file as it arrives, so a crash halfway through a long search doesn't lose everything. Module 5 covers files and JSON, and `dataclasses.asdict` from Module 3 converts a result to a dict.

6. **Timing decorator.** Add a `@timed` decorator to `fit` methods so you can see how training time compares to prediction time for each model. Where does k-NN spend its time?

7. **A scaling step for everything.** Instead of building scaling into one model, write a general `Scaled` wrapper that takes any model name and parameters, scales the data, and delegates to the inner model. Then you can search over `knn` with and without scaling too.

## What you used

- **`functools.cache`** for the data loader, returning immutable tuples so cached data can't be modified by accident
- **A registry decorator** so new models plug in without touching the search code
- **`Protocol`** to document the model interface without inheritance
- **`itertools.product`** to expand a grid, with an empty grid correctly giving one trial
- **Generators** (`expand_grid`, `all_trials`, `run_search`) so trials are created and results delivered lazily, one at a time
- **A `@contextmanager`** that times each trial and turns failures into recorded results, using `try`/`except`/`else`
- **A frozen dataclass** with a property for results
- **Inheritance and `super()`** to add scaling to logistic regression without repeating its training code
- **`sorted` with tuple keys** and **`itertools.groupby`** for the leaderboard
- **Type hints** with type aliases like `Dataset` and `Features`

And some ideas every ML practitioner needs: baselines, overfitting with small `k`, how feature scaling affects both accuracy and sensitivity to hyperparameters, the difference between validation and test sets, and why the test set is used only once.